In [1]:
import duckdb

In [2]:
con = duckdb.connect("research.duckdb")

In [3]:
con.execute("SHOW TABLES").df()

,name
0,current_eps_features
1,current_eps_raw
2,historical_eps_features
3,historical_news_clean
4,historical_prices_clean
5,master_developments
6,master_historical_eps
7,master_historical_prices
8,signal_price


### Get rid of news corresponding to iqids with no price data

In [4]:
con.execute("SELECT DISTINCT IQID FROM historical_prices_clean").df()

,IQID
0,7079247
1,4104164
2,4209952
3,133940468
4,6331068
...,...
3959,4208452
3960,4098794
3961,4660749
3962,102864


### Unnest the IQIDs list an create a row for each IQID in the news. Get rid of rows with no price data.

In [ ]:
def unlist_iqids():    
    con.execute("""
                CREATE OR REPLACE TABLE historical_news_clean AS
                WITH unnested AS (
                SELECT
                    dev_date,
                    CAST(UNNEST(IQIDS) AS INT) AS iqid,
                    dev_type,
                    headline,
                    description
                FROM master_developments
                )

                SELECT u.*
                FROM unnested u
                JOIN (SELECT DISTINCT iqid FROM historical_prices_clean) p
                ON u.iqid = p.iqid
    """)

### Clean up the news table, build a text input for LLM in the form "Company name: X (ticker: ABC), Headline: WW, Details: QQ"

In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE llm_data_input AS
    WITH clean_development_history AS (
    SELECT 
        DATE(
            CASE 
                WHEN d.dev_date LIKE '%:%' THEN
                    STRPTIME(d.dev_date, '%d/%m/%Y %H:%M:%S')
                ELSE
                    STRPTIME(d.dev_date, '%d/%m/%Y')
            END
        ) AS dev_date,
        d.iqid,
        SPLIT_PART(u.company_name, ' (', 1) AS clean_name,
        REPLACE(
            NULLIF(
                REGEXP_EXTRACT(u.company_name, '\\((?:.*:)?([A-Z\\. ]+)\\)', 1), ''), 
            ' ', '') AS ticker,        
        d.dev_type,
        d.headline,
        d.description            

    FROM historical_news_clean d
    JOIN (
        SELECT DISTINCT iqid
        FROM historical_prices_clean
    ) p
        ON d.iqid = p.iqid
    JOIN (
        SELECT iqid, ANY_VALUE(company_name) AS company_name
        FROM historical_eps_features
        GROUP BY iqid
    ) u
        ON d.iqid = u.iqid
    )
        
    SELECT
            dev_date,
            iqid,
            CONCAT(
                'Company: ', clean_name,
                ' (Ticker: ', ticker, '). ',
                'Headline: ', headline, '. ',
                'Details: ', description
            ) AS text_input
    FROM clean_development_history

""")

,Count
0,1652974


### Load the finbert model

In [90]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

tokenizer = AutoTokenizer.from_pretrained("yiyanghkust/finbert-tone")
model = AutoModelForSequenceClassification.from_pretrained("yiyanghkust/finbert-tone")

c:\Users\adamf\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "c:\Users\adamf\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\lib\c10.dll" or one of its dependencies.